In [26]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42

In [27]:
data_path = Path('./loan_approval_dataset.csv',skipinitialspace=True)
df = pd.read_csv(data_path)
df.columns = df.columns.str.strip()

df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [28]:
target_col = "loan_status"

df[target_col] = df[target_col].str.strip().str.lower()
df[target_col] = (df[target_col] == "approved").astype(int)

# IMPORTANT: Remove loan_amount from classifier to avoid leakage
X = df.drop(columns=[target_col, "loan_amount", "loan_id"])
y = df[target_col]

In [29]:
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

num_cols, cat_cols

(['no_of_dependents',
  'income_annum',
  'loan_term',
  'cibil_score',
  'residential_assets_value',
  'commercial_assets_value',
  'luxury_assets_value',
  'bank_asset_value'],
 ['education', 'self_employed'])

In [30]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

In [32]:
clf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=RANDOM_STATE))
])

param_dist = {
    "model__n_estimators": [200, 300, 400, 500],
    "model__max_depth": [None, 5, 10, 15],
    "model__min_samples_split": [2, 5, 10],
    "model__max_features": ["sqrt", "log2", None]
}

search = RandomizedSearchCV(
    clf_pipeline,
    param_dist,
    n_iter=20,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

search.fit(X_train, y_train)

best_clf = search.best_estimator_
print("Best Params:", search.best_params_)

Best Params: {'model__n_estimators': 400, 'model__min_samples_split': 5, 'model__max_features': 'sqrt', 'model__max_depth': 10}


In [33]:
y_pred = best_clf.predict(X_test)

print(classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.95      0.96       323
           1       0.97      0.98      0.98       531

    accuracy                           0.97       854
   macro avg       0.97      0.97      0.97       854
weighted avg       0.97      0.97      0.97       854

Confusion Matrix:
 [[306  17]
 [  8 523]]


==============================
🔵 STAGE 2 — REGRESSION
==============================

In [34]:
approved_df = df[df["loan_status"] == 1].copy()

reg_target = "loan_amount"

X_reg = approved_df.drop(columns=[reg_target, "loan_status", "loan_id"])
y_reg = approved_df[reg_target]

In [35]:
num_cols_reg = X_reg.select_dtypes(include=np.number).columns.tolist()
cat_cols_reg = X_reg.select_dtypes(include="object").columns.tolist()

In [36]:
reg_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols_reg),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols_reg)
])

In [37]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg,
    test_size=0.2,
    random_state=RANDOM_STATE
)

In [38]:
reg_pipeline = Pipeline([
    ("preprocessor", reg_preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE))
])

param_dist_reg = {
    "model__n_estimators": [200, 300, 400],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5, 10]
}

search_reg = RandomizedSearchCV(
    reg_pipeline,
    param_dist_reg,
    n_iter=15,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

search_reg.fit(Xr_train, yr_train)

best_reg = search_reg.best_estimator_
print("Best Regression Params:", search_reg.best_params_)

Best Regression Params: {'model__n_estimators': 300, 'model__min_samples_split': 10, 'model__max_depth': 5}


In [39]:
yr_pred = best_reg.predict(Xr_test)

rmse = np.sqrt(mean_squared_error(yr_test, yr_pred))

print("RMSE:", rmse)
print("MAE:", mean_absolute_error(yr_test, yr_pred))
print("R2:", r2_score(yr_test, yr_pred))

RMSE: 3285568.8253769455
MAE: 2451532.39827424
R2: 0.8744065949938311


In [40]:
joblib.dump(best_clf, "stage_1_classifier.pkl")
joblib.dump(best_reg, "stage_2_regressor.pkl")

['stage_2_regressor.pkl']

In [41]:
clf = joblib.load("stage_1_classifier.pkl")
reg = joblib.load("stage_2_regressor.pkl")

def two_stage_predict(applicant_df: pd.DataFrame):
    
    result = {}
    
    # Stage 1
    approve = clf.predict(applicant_df)[0]
    prob = clf.predict_proba(applicant_df)[0][1]
    
    result["approved"] = bool(approve)
    result["approval_probability"] = float(prob)
    
    # Stage 2
    if approve == 1:
        predicted_amount = reg.predict(applicant_df)[0]
        result["recommended_loan_amount"] = float(predicted_amount)
    
    return result

In [42]:
sample = pd.DataFrame([{
    "no_of_dependents": 1,
    "education": "Graduate",
    "self_employed": "No",
    "income_annum": 1200000,
    "loan_term": 12,
    "cibil_score": 820,
    "residential_assets_value": 2000000,
    "commercial_assets_value": 500000,
    "luxury_assets_value": 0,
    "bank_asset_value": 550000
}])

two_stage_predict(sample)

{'approved': True,
 'approval_probability': 0.89674300720465,
 'recommended_loan_amount': 3640673.5060712243}